# Color Jitter Augmentation Inspector

In [ ]:
import sys
sys.path.insert(0, '/Users/sofiahorlacher/VisualStudio/PASSION-Bias-Mitigation')

import os
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path


## Load Dataset Metadata

In [ ]:
dataset_dir = Path('/Users/sofiahorlacher/VisualStudio/PASSION-Bias-Mitigation/data')
meta_data_file = dataset_dir / 'label.csv'
image_folder_path = dataset_dir / 'PASSION'

df = pd.read_csv(meta_data_file, index_col=0)
print(f"Total subjects in metadata: {len(df)}")
print(f"Image folder path: {image_folder_path}")
print(f"Image folder exists: {image_folder_path.exists()}")

if image_folder_path.exists():
    num_images = len(os.listdir(image_folder_path))
    print(f"Total images in folder: {num_images}")

print(f"\nColumns in metadata: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

## Sample Random Images

In [ ]:
if image_folder_path.exists():
    image_files = [f for f in os.listdir(image_folder_path) if f.endswith('.jpg')]
    valid_image_files = [f for f in image_files if f.split('_')[0] in df.index]
    
    print(f"Found {len(valid_image_files)} images with matching subject IDs")
    
    listed_image_files = [
        'AA00970437_3.jpg',
        'AA00970507_1.jpg',
        'AA00971653_1.jpg',
    ]
    sample_image_files = [img_file for img_file in listed_image_files if img_file in valid_image_files]
    missing_image_files = [img_file for img_file in listed_image_files if img_file not in valid_image_files]
    
    if missing_image_files:
        print("Missing listed images:")
        for img_file in missing_image_files:
            print(f"- {img_file}")
    
    sample_data = []
    for img_file in sample_image_files:
        subject_id = img_file.split('_')[0]
        img_path = str(image_folder_path / img_file)
        row_data = df.loc[subject_id]
        
        if isinstance(row_data, pd.DataFrame):
            row_data = row_data.iloc[0]
        
        sample_data.append({
            'subject_id': subject_id,
            'img_path': img_path,
            'img_filename': img_file,
            'conditions_PASSION': row_data['conditions_PASSION'],
            'country': row_data['country'],
            'fitzpatrick': row_data['fitzpatrick'],
        })
    
    sample_df = pd.DataFrame(sample_data)
    
    print(f"\nSelected {len(sample_df)} listed images:")
    for idx, row in sample_df.iterrows():
        print(f"{idx+1}. {row['img_filename']} - {row['conditions_PASSION']}")
else:
    print(f"Image folder not found: {image_folder_path}")

## Define Color Augmentations at Different Strengths

In [ ]:
jitter_configs = {
    'Original': {
        'brightness': 0.0,
        'saturation': 0.0,
        'hue': 0.0,
    },

    'Conservative': {
        'brightness': 0.08,
        'saturation': 0.05,
        'hue': 0.0075,
    },

    'Safe': {
        'brightness': 0.14,
        'saturation': 0.10,
        'hue': 0.015,
    },

    'Balanced': {
        'brightness': 0.20,
        'saturation': 0.14,
        'hue': 0.025,
    },

    'Moderate-Aggressive': {
        'brightness': 0.26,
        'saturation': 0.18,
        'hue': 0.035,
    },

    'Aggresive': {
        'brightness': 0.32,
        'saturation': 0.22,
        'hue': 0.045,
    }
}

def make_display_name(name, params, bound='upper'):
    if name == 'Original':
        return 'Original'

    sign = -1 if bound == 'lower' else 1
    brightness_factor = max(0, 1 + sign * params['brightness'])
    saturation_factor = max(0, 1 + sign * params['saturation'])
    hue_factor = max(0, 1 + sign * params['hue'])
    bound_label = 'Lower' if bound == 'lower' else 'Upper'
    return (
        f"{name} ({bound_label})\n"
        f"B={brightness_factor:.2f} S={saturation_factor:.2f}\n"
        f"H*={hue_factor:.3f}"
    )

def apply_fixed_jitter(img, params, bound='upper'):
    if params['brightness'] == params['saturation'] == params['hue'] == 0.0:
        return img.copy()

    sign = -1 if bound == 'lower' else 1
    brightness_factor = max(0, 1 + sign * params['brightness'])
    saturation_factor = max(0, 1 + sign * params['saturation'])
    hue_factor = max(0, 1 + sign * params['hue'])

    rgb = np.array(img.convert('RGB'))
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV).astype(np.float64)

    hsv[:, :, 0] = hsv[:, :, 0] * hue_factor
    hsv[:, :, 1] = hsv[:, :, 1] * saturation_factor
    hsv[:, :, 2] = hsv[:, :, 2] * brightness_factor
    hsv[hsv > 255] = 255

    jittered_bgr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    jittered_rgb = cv2.cvtColor(jittered_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(jittered_rgb)

display_configs_lower = []
display_configs_upper = []
for name, params in jitter_configs.items():
    display_configs_lower.append((make_display_name(name, params, bound='lower'), params))
    display_configs_upper.append((make_display_name(name, params, bound='upper'), params))

print('OpenCV HSV jitter configurations defined:')
for name, params in jitter_configs.items():
    print(
        f"  {name}: "
        f"brightness factor [{max(0, 1 - params['brightness']):.2f}, {1 + params['brightness']:.2f}], "
        f"saturation factor [{max(0, 1 - params['saturation']):.2f}, {1 + params['saturation']:.2f}], "
        f"hue factor [{max(0, 1 - params['hue']):.3f}, {1 + params['hue']:.3f}]"
    )


## Visualize Augmentations for Each Sample Image

In [ ]:
for sample_idx, (i, row) in enumerate(sample_df.iterrows()):
    img_path = row['img_path']

    if not os.path.exists(img_path):
        print(f"Image not found: {img_path}")
        continue

    try:
        img = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        continue

    for bound_name, display_configs in [('Lower Bounds', display_configs_lower), ('Upper Bounds', display_configs_upper)]:
        fig, axes = plt.subplots(2, 3, figsize=(17, 10))
        fig.suptitle(f'{bound_name} using OpenCV HSV scaling', fontsize=14, fontweight='bold')
        axes = axes.flatten()

        for ax_idx, (config_name, params) in enumerate(display_configs):
            bound = 'lower' if 'Lower' in config_name else 'upper'
            aug_img = apply_fixed_jitter(img, params, bound=bound)

            axes[ax_idx].imshow(aug_img)
            axes[ax_idx].set_title(config_name, fontsize=10, fontweight='bold')
            axes[ax_idx].axis('off')

        plt.tight_layout()
        plt.show()
    print()
